In [ ]:
import mg5qs_imports as qs
from pathlib import Path
import os
import numpy as np

### Changing parameters across two dimensions

This example demonstrates how to change paramaters across two (or more) dimensions.

This example includes:
- setting multiple parameters independently
- generating and showering events
- reassembling results in a coherent order
- plotting results across two dimensions

### Choose a process that generates $\tau$ particles

namely: <code>generate p p > ta- vt~</code>

In [ ]:
INPUT_PATH = Path.cwd()/'input' # madgraph cards
# note that for this example, we move to a process which directly generates taus
qs.edit_card(INPUT_PATH, card_name='proc_card_example4.dat')

In [ ]:
output_name, FRAMEWORK_PATH = qs.run_MG5(INPUT_PATH, proc_card_name='proc_card_example4.dat')

In [ ]:
qs.edit_card(FRAMEWORK_PATH)

### Retrieve standard values

This example varies $M_Z$ and $m_{\tau}$:

In [ ]:
card = qs.ParamCard(FRAMEWORK_PATH / 'Cards' / 'param_card.dat')
card.dfs()['MASS'] # standard values for mass are returned by MadGraph

In [ ]:
from IPython.display import Math, display

# side trip for fun fact: MadGraph comments show how certain values are calculated
W_id = 24
df_masses = card.dfs()['MASS']
df_masses[df_masses['key'] == W_id]['comment'].item() # how is mass for W+ calculated?

### Set values across two parameters

First, retrieve standard values and create 5 test parameters

In [ ]:
# want to vary both masses in a range where the physics is highly sensitive -- even if values are unreasonable
Z_id = 23
tau_id = 15
card = qs.ParamCard(FRAMEWORK_PATH / 'Cards' / 'param_card.dat')
m_z, m_tau = card.get_value('MASS', Z_id), card.get_value('MASS', tau_id) # retrieve standard values
MTAU = np.linspace(1,100,5)*1.777 #vary over a wide range of values
MZ = np.linspace(1,10,5)*91.188   #vary over a wide range of values
MTAU, MZ

Second, generate LHEs for resulting 5x5 pairs of parameter values.

**Important! MadGraph will produce LHEs in no particular order**

In [ ]:
# generate one set of LHE events for each combination of parameter values
card = qs.ParamCard(FRAMEWORK_PATH / 'Cards' / 'param_card.dat')
for mtau in MTAU:
    card.set_value('MASS', tau_id, mtau)
    for mz in MZ:
        print(f'Generating LHE with: mtau = {mtau} mz = {mz}')
        card.set_value('MASS', Z_id, mz)
        qs.generate_LHE(card, FRAMEWORK_PATH)

Third, shower all resulting LHEs. These are independent of each other, so order is not important.

In [ ]:
qs.pythia_parallel([tau_id], FRAMEWORK_PATH, 'EXAMPLE_CONTOUR', topics='P_mu', size=1000000)

### Retrieve all files in a dictionary (contact=False)

In this use case, results are independent, representing a run for a unique pair of parameters. The <code>qs.unpickle</code> returns a dictionary of results keyed to run name:

In [ ]:
data = qs.unpickle('EXAMPLE_CONTOUR', concat=False)
data.keys()

### Add $p_{\tau}$ to each set of results:

Each dictionary entry in <code>data</code> contains a tuple (parameters, results) of data type (ParamCard, np.array):

In [ ]:
next(iter(data.values())) # entry entry contains a tuple (ParamCard, results)

Calculate $p_{\tau}$ for each set of results:

In [ ]:
for v in data.values(): # compute pT col in set of results 
    v[1]['pT'] = np.sqrt((v[1]['px']**2)+(v[1]['py']**2)) #v[0] is ParamCard, v[1] is results

### Graph results (in a rational order)

In [ ]:
from scipy import stats
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
fig, axes = plt.subplots(5, 5, figsize=(16, 12))
axes = axes.flatten()
BINS = np.linspace(0, 500, 30)

# Madgraph produces results in arbitrary order
# inspect each parameter setting to re-establish original order
rows = []
for k in data.keys():
    df_masses = data[k][0].dfs()['MASS']
    m_tau = df_masses[df_masses['key'] == tau_id]['value'].item() # setting for mass of tau
    m_z = df_masses[df_masses['key'] == Z_id]['value'].item() # setting for mass of W
    rows.append((k,m_tau,m_z))
df_runs = pd.DataFrame(rows, columns=['run', 'm_tau', 'm_z'])
df_runs.sort_values(['m_tau','m_z'], inplace = True) # original order
ordered_runs = df_runs['run']

for idx, run in enumerate(ordered_runs):
    pc_df = data[run]
    ax = axes[idx]
    ax.hist(pc_df[1]['pT'], bins=BINS)
    m_tau = pc_df[0].get_value('MASS', tau_id)
    m_z = pc_df[0].get_value('MASS', Z_id)
    ax.set_title(f"$m_\\tau$ = {m_tau:.3f},   $m_z$ = {m_z:.3f}")
    ax.set_xlabel('GeV')
    ax.set_yscale('log')

plt.tight_layout()
plt.show()